# Import Statements

In [140]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import yfinance as yf

from modules.utils import *
from modules.screen import *
from modules.data_loader import *
from modules.portfolio import *
from modules.trade import *
from modules.backtest import *
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from modules.tearsheet import Tearsheet

import plotly.io as pio
pio.renderers.default = "notebook_connected"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


---
# Parameters

In [141]:
config = load_config("config.yaml")

# Extract config parameters
start_date = config["start_date"]
end_date = config["end_date"]
BENCHMARK = config["benchmark"]
UPDATE = config["update"]
SCREEN = config["screen"]

---
# Data Fetching

In [142]:
# Loading data for all NSE tickers and calculating features
full_nse_tickers = load_nse_all()
load_full_data = load_data(start_date=start_date, end_date=end_date, update=UPDATE, full_nse_tickers=full_nse_tickers,
                           benchmark=BENCHMARK)
# load benchmark data
benchmark_prices = download_benchmark_data(symbol=BENCHMARK)

# Forward fill missing price data for each ticker
df = load_full_data.copy().reset_index()
df["Date"] = pd.to_datetime(df["Date"])

priceData = (
    df.pivot(index="Date", columns="TIC", values=["Adj Close", "Open"])
      .sort_index()
      .ffill()
)

priceData = (
    priceData
    .stack(level=1)
    .reset_index()
)

priceData.columns = ["Date", "TIC", "Adj Close", "Open"]

---
# Backtest - Simulation

In [143]:
backtest_df, trade_blotter, portfolio_history = run_backtest(
    price_df=priceData,
    screen_rule=SCREEN,
    initial_cash=config["cash"],
    rebalance_freq=config["rebalance_frequency"],
    feature_df=load_full_data,
    start_date=start_date,    
    end_date=end_date,
    allocator="max_sharpe",
    ranker_dict=config["ranker"],
    config=config,
)


📅 Progress: 0/801 | Date: 2023-01-02

📅 Progress: 10/801 | Date: 2023-01-16

📅 Progress: 20/801 | Date: 2023-01-31

🔁 Rebalance triggered on 2023-01-31
   💰 Portfolio value: 1,000,000.00
   🧾 Cash: 1,000,000.00
   🎯 Screened stocks: 106
   📊 Ranked stocks: 10
   🧠 Top allocations: [('JWL.NS', 0.69536), ('OIL.NS', 0.12771), ('GRAVITA.NS', 0.05611), ('JSL.NS', 0.04483), ('IOC.NS', 0.0391)]
   💼 Trades executed | New cash: 150.98

📅 Progress: 30/801 | Date: 2023-02-14

📅 Progress: 40/801 | Date: 2023-02-28

🔁 Rebalance triggered on 2023-02-28
   💰 Portfolio value: 790,295.92
   🧾 Cash: 150.98
   🎯 Screened stocks: 103
   📊 Ranked stocks: 10
   🧠 Top allocations: [('CIGNITITEC.NS', 0.21423), ('ZFCVINDIA.NS', 0.14419), ('CRISIL.NS', 0.13941), ('BLUESTARCO.NS', 0.12761), ('ITC.NS', 0.12253)]
   💼 Trades executed | New cash: 6,581.18

📅 Progress: 50/801 | Date: 2023-03-15

📅 Progress: 60/801 | Date: 2023-03-29

🔁 Rebalance triggered on 2023-03-31
   💰 Portfolio value: 776,157.17
   🧾 Cash: 6

---
# Portfolio Analysis

In [144]:
pv = backtest_df.copy()
pv["Date"] = pd.to_datetime(pv["Date"])
pv = pv.set_index("Date")["equity"]

bv = benchmark_prices.copy()
bv["Date"] = pd.to_datetime(bv["Date"])
bv = bv.set_index("Date")["bench"]

# align with portfolio dates
bv = bv.reindex(pv.index).ffill()

In [148]:
ts = Tearsheet(
    portfolio_values = pv,
    benchmark_values = bv,
    trades = trade_blotter,   # ignore trades for now,
    portfolio_history=portfolio_history,
    risk_free_rate = 0.06
)

# All stats as Plotly tables (show inline in Jupyter/VSCode)
tables = ts.summary()
display(tables["summary"])

# Everything at once
ts.plot_all()

,Value
Start Date,2023-01-02
End Date,2026-04-02
N Days,801.00
N Years,3.25
Total Return,152.27%
CAGR,32.97%
Volatility,31.89%
Sharpe,0.89
Sortino,1.20
Calmar,0.81



══════════════════════════════════════════════════════════════
  PORTFOLIO TEARSHEET
  2023-01-02  →  2026-04-02  (801 days / 3.2 yrs)
══════════════════════════════════════════════════════════════

